# Day 4 (Actually I skipped a day)

Till here we found out that the neural networks are nothing just a set of 3 things, forward pass, backward pass, and graident descent. And we do it again and again on many inputs.

#### We will try to redefine our NN this time not using an overhead of unit()

First, why this approach is not optimized, \
we are using this line  \
```self.weights = [unit(random.uniform(-1, 1), symbol = f'w{i}')for i in range(n_inputs)]``` \
to declare and define weights which is good, but it is calling unit class a python overhead, and als we are unable to use the benifit of using lists, \
lists gives you a contiguous array which is helpful as when cpu get a value from memory it gets a bunch of memory and save it into cache as a prediction that we will use those values, \
but when we are using weight.data in \
```value = sum((weight*inputs[i] for i,weight in enumerate(self.weights)), self.bias)``` \
these are not stored in contiguous fashion instead randomly, as we have pointers in the list instead of values.\
So now we will go with a different approach of using weights as a float array instead of using units\
However for that we will need to arrange some things, it is a design problem.\
How to design a neuron now, how we will calculate grads and store them, and what to store in a neuron to do all operations \
I did some rough work which you can see, I will make a better version later


In [ ]:
from IPython.display import Image
Image(filename='/Users/saumyasharma/Downloads/IMG_1608.png', width=300, height=300) 

In [ ]:
Image(filename='/Users/saumyasharma/Downloads/IMG_1609.png', width=300, height=300) 

Forward pass \
Q = sum(xi*wi) + b -> as mentioned in photo dout/dQ = f'(Q) -> so I need to save this \
f(activation function) will be used to find output

Backward pass \
We will have a grad_in = dL/dout \
We will calculate dL/dQ = grad_in * dout /dQ  = grad_in * f'(Q) \
dL/dwi = dL/dQ * dQ/dwi = grad_in * f'(Q) * xi\
dL/db = grad_in * f'(Q) * 1\
dL/dxi = grad_in * f'(Q) * wi

So from all these calculations I got that we need to save these and also some I will talk about next
1. weight list
2. bias
3. input list
4. Q


Let's design Layer

In [ ]:
import random
import math

In [ ]:
class Neuron:
    def __init__(self, n_inputs, activation='ReLU'):
        self.weights = [random.uniform(-1, 1) for i in range(n_inputs)]
        self.weights_grad = [0] * n_inputs
        self.bias = 0
        self.bias_grad = 0
        self.activation = activation
        self.Q = 0

    def forward(self, input_layer):
        self.Q = sum((x * y for x, y in zip(self.weights, input_layer)) , self.bias)
        if self.activation == 'tanh':
            output_value = self.Q.tanh()
        elif self.activation == 'relu':
            output_value = self.Q.ReLU()
        else:
            output_value = self.Q
        return output_value
        

In [ ]:
neuron1 = Neuron(3)

In [ ]:
# forward loop 
x_input = [1.0, -1.0, 1.0]
output_layer1 = layer1.forward(x_input)


In [ ]:
output_layer1

In [ ]:
class Layer:
    def __init__(self, n_neurons, n_inputs, activation='ReLU'):
        self.neurons = [Neuron(n_inputs,activation) for i in range(n_neurons)]

    def forward(self, input_layer):
        output = [neuron.forward(input_layer) for neuron in self.neurons]
        return output

In [ ]:
layer1 = Layer(3,3)

In [ ]:
layer2 = Layer(4,3)

In [ ]:
layer3 = Layer(1,4)

In [ ]:
layer3.forward((layer2.forward(layer1.forward(x_input))))

In [ ]:
layer3.neurons

In [ ]:
class MLP:
    def __init__(self, layer_list, activation_list='ReLU'):
        self.layers = [Layer(layer_list[i], layer_list[i-1]) for i in range(1,len(layer_list))]
    def forward(self, inputs):
        output = inputs
        for layer in self.layers:
            output = layer.forward(output)
        return output

In [ ]:
nn = MLP([3,3,4,1])

In [ ]:
nn.forward(x_input)

# Day 5

#### Now we will add backward pass to all, first for that we need to do one thing which is thinking of how we can do the backward pass without a smaller python object

In [302]:
class Neuron:
    
    def __init__(self, n_inputs, activation='ReLU'):
        self.weights = [random.uniform(-1, 1) for i in range(n_inputs)]
        self.weights_grad = [0] * n_inputs
        self.bias = 0
        self.bias_grad = 0
        self.activation = activation
        self.Q = 0
        self.input_value = None

    def forward(self, input_layer):
        self.input_value = input_layer
        self.Q = sum((x * y for x, y in zip(self.weights, input_layer)) , self.bias)
        if self.activation == 'tanh':
            output_value = math.tanh(self.Q)
        elif self.activation == 'ReLU':
            output_value = max(self.Q, 0)
        else:
            output_value = self.Q
        return output_value

    def backward(self, grad_in):
        dfdq = 1
        if self.activation == 'ReLU':
            dfdq = 1 if self.Q > 0 else 0
        elif self.activation == 'tanh':
            dfdq = 1 - math.tanh(self.Q)**2
        dLdQ = grad_in * dfdq
        dLdb = dLdQ
        self.bias_grad += dLdb
        dLdW = [dLdQ * i for i in self.input_value]
        for i in range(len(self.weights_grad)):
            self.weights_grad[i] += dLdW[i]
        return [dLdQ * w for w in self.weights]

    # below are just for testing not used in our current setup 
    # def parameters(self):
    #     return self.weights + [self.bias]

    # def grads(self):
    #     return self.weights_grad + [self.bias_grad]

    # def output(self):
    #     return [self.Q]

class Layer:
    
    def __init__(self, n_neurons, n_inputs, activation='ReLU'):
        self.neurons = [Neuron(n_inputs,activation) for i in range(n_neurons)]
        self.input_layer_neurons = n_inputs

    def forward(self, input_layer):
        output = [neuron.forward(input_layer) for neuron in self.neurons]
        return output

    def backward(self, grad_in_list):
        # for each neuron there will be a grad 
        # each neuron will output a list of dL/dX -> we will add them all
        dLdX = [0] * self.input_layer_neurons
        for neuron,grad_in in zip(self.neurons, grad_in_list):
            delta = neuron.backward(grad_in)
            dLdX = [dLdXi + delta_i for dLdXi, delta_i in zip(dLdX, delta)]
        return dLdX

    # below are just for testing not used in our current setup 
    # def parameters(self):
    #     parameters = []
    #     for neuron in self.neurons:
    #         parameters += neuron.parameters()
    #     return parameters

    # def grads(self):
    #     grads = []
    #     for neuron in self.neurons:
    #         grads += neuron.grads()
    #     return grads

    # def output(self):
    #     outputs = []
    #     for neuron in self.neurons:
    #         outputs += neuron.output()
    #     return outputs
        

class MLP:
    
    def __init__(self, layer_list, activation='ReLU'):
        self.layers = [Layer(layer_list[i], layer_list[i-1], activation) for i in range(1,len(layer_list))]
        
    def forward(self, inputs):
        output = inputs
        for layer in self.layers:
            output = layer.forward(output)
        return output

    def backward(self, loss_grad):
        grad_in = loss_grad
        for layer in reversed(self.layers):
            grad_in = layer.backward(grad_in)

    # below are just for testing not used in our current setup 
    # def parameters(self):
    #     parameters = []
    #     for layer in self.layers:
    #         parameters += layer.parameters()
    #     return parameters

    # def grads(self):
    #     grads = []
    #     for layer in self.layers:
    #         grads += layer.grads()
    #     return grads

    # def output(self):
    #     outputs = []
    #     for layer in self.layers:
    #         outputs += layer.output()
    #     return outputs

In [303]:
nn = MLP([2,3,3,1], 'tanh')

In [304]:
inputs = [[1.0,2.0],[-1.0,2.0],[-2.0,1.0]]
y_true = [3.0, 1.0, -1.0]

In [307]:
y_pred = nn.forward(inputs[0])
y_pred

[-0.33093017078612946]

In [309]:
Loss = (y_pred[0]-y_true[0])**2
Loss

11.095095802653315

In [310]:
dLossdy_pred = 2*(y_pred[0]-y_true[0])*1
dLossdy_pred

-6.661860341572259

In [311]:
nn.backward([dLossdy_pred])

We have implemented the model part but now we need to make loss function which can work directly with floats

In [312]:
class MSE:
    
    def __init__(self, y_pred, y_true):
        self.y_pred = y_pred
        self.y_true = y_true
        self.inputs = len(y_pred)

    def loss(self):
        ans = 0.0
        grads = []
        for y_predi, y_truei in zip(self.y_pred,self.y_true):
            ans += (y_predi-y_truei)**2 
            grads.append(2*(y_predi-y_truei))
        ans /= self.inputs
        for i in range(self.inputs):
            grads[i]/=self.inputs
        return (ans, grads)
        
        
                
        
    

In [313]:
loss_func = MSE(y_pred, [y_true[0]])

In [314]:
loss, loss_grad =  loss_func.loss()

In [315]:
loss

11.095095802653315

In [316]:
loss_grad

[-6.661860341572259]

In [317]:
nn.backward(loss_grad)

# Day 6

### How we are going to implement optimizers

Now it is the time to implement optimizer SGD, but for this we will need to restructure our code,\
why??\
..\
..\
We are getting list of parameters when call nn.parameters() but it is the copy, instead we need the original list so that we can chnage the parameters values\
for that we will now try to implement such that grad and parameters are coming by reference.\
Why grad??\
..\
..\
for optimizer.zero_grad()


#### Implementing call by reference for parameters and grad 
For this I am combining weights and bias into a single list

In [319]:
class Neuron:
    
    def __init__(self, n_inputs, activation='ReLU'):
        self.params = [random.uniform(-1, 1) for i in range(n_inputs+1)]
        self.params_grad = [0] * (n_inputs+1)
        self.activation = activation
        self.Q = 0
        self.input_value = None
        self.n = n_inputs + 1 # +1 as we will have n weights and 1 bias, and you will see next that we are adding 1 in inputs as well

    def forward(self, input_layer):
        # we are adding this as bias needs to be added in the self.Q which is equivalent to self.params[-1] * 1
        self.input_value = input_layer + [1.0] 
        self.Q = sum((x * y for x, y in zip(self.params, self.input_value)))
        if self.activation == 'tanh':
            output_value = math.tanh(self.Q)
        elif self.activation == 'ReLU':
            output_value = max(self.Q, 0)
        else:
            output_value = self.Q
        return output_value

    def backward(self, grad_in):
        dfdq = 1
        if self.activation == 'ReLU':
            dfdq = 1 if self.Q > 0 else 0
        elif self.activation == 'tanh':
            dfdq = 1 - math.tanh(self.Q)**2
        dLdQ = grad_in * dfdq
        for i in range(self.n):
            self.params_grad[i] += dLdQ * self.input_value[i]
        # why this -1?? Because we have only n input neurons, n+1 is for bias term we added
        return [dLdQ * w for w in self.params[:-1]] 
        

    def parameters(self):
        return self.params

    def grads(self):
        return self.params_grad


class Layer:
    
    def __init__(self, n_neurons, n_inputs, activation='ReLU'):
        self.neurons = [Neuron(n_inputs,activation) for i in range(n_neurons)]
        self.input_layer_neurons = n_inputs

    def forward(self, input_layer):
        output = [neuron.forward(input_layer) for neuron in self.neurons]
        return output

    def backward(self, grad_in_list):
        # for each neuron there will be a grad 
        # each neuron will output a list of dL/dX -> we will add them all
        dLdX = [0] * self.input_layer_neurons
        for neuron,grad_in in zip(self.neurons, grad_in_list):
            delta = neuron.backward(grad_in)
            dLdX = [dLdXi + delta_i for dLdXi, delta_i in zip(dLdX, delta)]
        return dLdX

    # we are getting list of parameters by refrence and here as well we need to pass by refrence
    def parameters(self):
        parameters = []
        for neuron in self.neurons:
            parameters.append(neuron.parameters())
        return parameters

    def grads(self):
        grads = []
        for neuron in self.neurons:
            grads.append(neuron.grads())
        return grads
        

class MLP:
    
    def __init__(self, layer_list, activation='ReLU'):
        self.layers = [Layer(layer_list[i], layer_list[i-1], activation) for i in range(1,len(layer_list))]
        
    def forward(self, inputs):
        output = inputs
        for layer in self.layers:
            output = layer.forward(output)
        return output

    def backward(self, loss_grad):
        grad_in = loss_grad
        for layer in reversed(self.layers):
            grad_in = layer.backward(grad_in)

    def parameters(self):
        parameters = []
        for layer in self.layers:
            parameters.append(layer.parameters())
        return parameters

    def grads(self):
        grads = []
        for layer in self.layers:
            grads.append(layer.grads())
        return grads
        
        
        
            
        

In [320]:
nn = MLP([2,3,3,1], 'tanh')

In [321]:
nn.parameters()

[[[-0.8108927556802614, 0.6880985465394693, 0.6897759394282881],
  [0.9924688305067926, -0.24960714746000567, -0.5288204542288408],
  [0.5508648230139144, 0.1690587652060953, 0.32994109620908185]],
 [[0.8426268007166025,
   -0.6848419190724071,
   -0.5390768546041873,
   0.8073449746256114],
  [-0.1938512384711255,
   0.458562689373311,
   0.06559809472379641,
   0.7368493675470236],
  [-0.2083633931183344,
   -0.04282035550499064,
   -0.44229588838834877,
   0.5083102985779548]],
 [[0.41438405167893544,
   0.11091775250090197,
   -0.8770527803654342,
   -0.6015424205280295]]]

In [323]:
nn.grads()

[[[0, 0, 0], [0, 0, 0], [0, 0, 0]],
 [[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]],
 [[0, 0, 0, 0]]]

In [327]:
class MSE:
    def __init__(self):
        pass

    def loss(self, y_pred, y_true):
        inputs = len(y_pred)
        ans = 0.0
        grads = []
        for y_predi, y_truei in zip(y_pred,y_true):
            ans += (y_predi-y_truei)**2 
            grads.append(2*(y_predi-y_truei))
        ans /= inputs
        for i in range(inputs):
            grads[i]/=inputs
        return (ans, grads)

In [328]:
class SGD:
    def __init__(self, parameters, grads, learning_rate=0.01):
        self.parameters = parameters
        self.grads = grads
        self.learning_rate = learning_rate

    def step(self):
        for layer_parameters, layer_grads in zip(self.parameters, self.grads):
            for neuron_parameters, neuron_grad in zip(layer_parameters, layer_grads):
                for i in range(len(neuron_parameters)):
                    neuron_parameters[i] -= self.learning_rate * neuron_grad[i]
                    
        
    def zero_grad(self):
        for layer_grads in self.grads:
            for neuron_grads in layer_grads:
                 for i in range(len(neuron_grads)):
                    neuron_grads[i] = 0
        
        

### We will try to train xor and see if it is getting trained

In [338]:
# For XOR
inputs = [
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
]
targets = [[0.0], [1.0], [1.0], [0.0]]

In [339]:
# Training loop
for each epoch:
    total_loss = 0
    for each inputs:
        output = nn.forward(input)
        loss, loss_grad = mse.loss(output, y_true)
        nn.backward(loss_grad)
        optimizer.step()
        optimizer.zero_grad()

SyntaxError: invalid syntax (2900688640.py, line 2)

In [340]:
nn = MLP([2,4,4,1], 'tanh')

In [341]:
loss_func = MSE()

In [342]:
optimizer = SGD(nn.parameters(), nn.grads(), 0.05)

In [343]:
epochs = 100

In [344]:
# Training
for epoch in range(epochs):
    total_loss = 0
    for input_value, target in zip(inputs, targets):
        output = nn.forward(input_value)
        loss, loss_grad = loss_func.loss(output, target)
        total_loss += loss
        nn.backward(loss_grad)
        optimizer.step()
        optimizer.zero_grad()
    if epoch%10 == 0:
        print(f"epoch {epoch}, total loss: {total_loss}")

epoch 0, total loss: 2.0255156894684836
epoch 10, total loss: 1.0809471642189608
epoch 20, total loss: 0.9292876934651051
epoch 30, total loss: 0.8292661221560464
epoch 40, total loss: 0.7490336058336847
epoch 50, total loss: 0.6430203706610822
epoch 60, total loss: 0.4382072731950063
epoch 70, total loss: 0.17229120782952442
epoch 80, total loss: 0.06709027687259288
epoch 90, total loss: 0.036462871277694195


In [345]:
output = nn.forward([1.0, 1.0])

In [346]:
output

[-0.020986751435741767]